## Overview

## Setup

In [44]:
%load_ext dotenv
%dotenv

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [1]:
!pip install \
  langchain==0.3.27 \
  langchain-pinecone==0.2.11 \
  langchain-openai==0.3.30 \
  langchain-mcp-adapters==0.1.9 \
  langgraph==0.6.6

Looking in indexes: https://pypi.org/simple, http://nxd-pip-registry/index/
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 787.8/787.8 kB 33.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.6/587.6 kB 25.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 54.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 70.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 86.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.9/798.9 kB 32.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 111.7 MB/s  0:00:00
  Attempting uninstall: packaging━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  8/64 [python-dotenv]
    Found existing installation: packaging 25.0━━━━━━━━━━━━━━━  8/64 [python-dotenv]
    Uninstalling packaging-25.0:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  8/64 [python-dotenv]
      Successfully uninstalled packaging-25.0━━━━━━━━━━━━━━━

In [38]:
from os import getenv

In [39]:
import nxd.data_product.context as ctx
from nxd.data_product.client import create_client

Loading nxd.data_product v0.1.4


In [41]:
from langchain_core.tools.retriever import create_retriever_tool
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_openai import ChatOpenAI
from langchain_pinecone import PineconeEmbeddings, PineconeVectorStore
from langgraph.prebuilt import create_react_agent

## Nextdata

In [ ]:
nxd_client = create_client(hostname="dp.demo.trynxd.com")
data_product = nxd_client.data_product(data_product="au-competitor-analysis-demo")
pinecone_output = data_product.get_outputs("pinecone", ctx.PineconeOutput)

## Pinecone - Vector Store

In [ ]:
search_configuration = {
    "namespace": "au-competitor-analysis-documents",
    "k": 100,
    "fetch_k": 500,
}

embeddings = PineconeEmbeddings(
    model="llama-text-embed-v2",
    api_key=pinecone_output.password,  # type: ignore
)

retriever = PineconeVectorStore(
    index_name="finance-documents-pilot",
    embedding=embeddings,
    namespace=pinecone_output.namespace,
    pinecone_api_key=pinecone_output.password,
).as_retriever(search_kwargs=search_configuration)

retriever_tool = create_retriever_tool(
    retriever,
    name="documents",
    description="Announcement documents from NASDAQ listed companies",
)

## Nextdata Data Products - MCP

In [ ]:
data_product_mcps = MultiServerMCPClient(
    {
        "product-competitiveness": {
            "url": "https://dp.demo.trynxd.como/rpcs/mcp-api/mcp/",
            "transport": "streamable_http",
            "headers": {
                "x-nextdata-token": getenv("NXD_PAT"),
            },
        },
    }
)

## OpenAI - LLM

In [46]:
llm = ChatOpenAI(
    openai_api_key=getenv("OPENAI_API_KEY"),  # type: ignore
    model="gpt-4o-mini",
    temperature=0.0,
)

## Agent

In [47]:
tools = [
    *await data_product_mcps.get_tools(),
    retriever_tool,
]

agent = create_react_agent(
    llm,
    tools,
)

content = "Provide income insights from Westpac's 3Q25 Update"
await agent.ainvoke({"messages": [{"role": "user", "content": content}]})

{'messages': [HumanMessage(content="Provide income insights from Westpac's 3Q25 Update", additional_kwargs={}, response_metadata={}, id='0f2835c5-ef88-4494-ab0e-5971845f9adb'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_VRw544iAgjDQd9n0SNufLKwI', 'function': {'arguments': '{"query":"Westpac 3Q25 Update"}', 'name': 'documents'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 98, 'total_tokens': 117, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_560af6e559', 'id': 'chatcmpl-C6zu13ST6vPA5Gnh0teAti6m0Aua8', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--22329d31-4367-4ab9-a5ca-b197367e919c-0', tool_calls=[{'name': 'documents', 'args'